# Part 12 — The Eval Flywheel

*Evals from First Principles, Part 12.*

Your best eval set is not something you design once in a vacuum, it is mined from production. This notebook takes 12 mock traces from a support assistant. Each trace carries CHEAP signals we get for free (the model's self-reported confidence and a guardrail / thumbs-down flag), but its true label stays HIDDEN until a human spends an annotation on it. We rank the traces by a cheap priority score, spend a small annotation budget on the most suspicious ones, reveal their labels, and watch a golden set grow exactly where the model is weakest, round by round.

Pure Python, no imports beyond the standard library, fully offline and deterministic (fixed seed for the random comparison) — every number below is reproducible on a laptop with no network and no API key.

Companion script: `flywheel.py`


In [ ]:
# --- The production log -------------------------------------------------
# 12 traces served by a support assistant. confidence = the model's self-report
# (cheap, always visible). flag = a guardrail hit or a thumbs-down (cheap,
# always visible). correct = the HIDDEN ground truth (1 = good answer,
# 0 = a failure) that a human reveals ONLY when we spend budget to annotate it.
import random
from collections import namedtuple

Trace = namedtuple("Trace", "id question confidence flag correct")

TRACES = [
    Trace("T01", "How do I reset my password?",        0.96, 0, 1),
    Trace("T02", "What are your business hours?",       0.93, 0, 1),
    Trace("T03", "Where is my refund?",                 0.91, 0, 1),
    Trace("T04", "Change my shipping address",          0.88, 0, 1),
    Trace("T05", "Cancel my subscription",              0.84, 0, 1),
    Trace("T06", "Do you ship to Canada?",              0.80, 0, 1),
    Trace("T07", "Is the pro plan tax-deductible?",     0.74, 0, 1),
    Trace("T08", "Why was I charged twice?",            0.68, 0, 0),
    Trace("T09", "Explain your data-retention policy",  0.62, 1, 0),
    Trace("T10", "Can I get a discount?",               0.55, 1, 1),
    Trace("T11", "Is my warranty still valid?",         0.48, 1, 0),
    Trace("T12", "Reactivate my closed account",        0.40, 1, 0),
]

print(f"{len(TRACES)} traces. conf and flag are free; the true label is hidden.")
for t in TRACES:
    print(f"  {t.id}  conf={t.confidence:.2f}  flag={t.flag}  (label hidden)")


## Step 1 — A cheap priority score

We cannot see the true label without paying for it, but we already have two free signals for every trace: the model's own confidence, and whether a guardrail or a thumbs-down fired. Low confidence and a flag both point the same way, toward a trace worth checking. We fold them into a single suspicion score:

`priority = (1 - confidence) + 0.5 * flag`

No annotation required to compute it. It is a bet about where the failures are, not a measurement of them.


In [ ]:
def priority(t):
    """A cheap suspicion score: low confidence and a flag both push it up.

    priority = (1 - confidence) + 0.5 * flag. Needs NO annotation to compute."""
    return (1 - t.confidence) + 0.5 * t.flag


print("  id    conf  flag  priority")
for t in TRACES:
    print(f"  {t.id}   {t.confidence:>5.2f}  {t.flag:>3}   {priority(t):>6.2f}")

ranked = sorted(TRACES, key=priority, reverse=True)
print()
print("Ranked by priority (highest = most suspicious), no labels needed yet:")
print("  " + "  ".join(t.id for t in ranked))
# T12  T11  T10  T09  T08  T07  T06  T05  T04  T03  T02  T01


## Step 2 — Spend the budget: hard-case vs random

Ranking is free, but checking a label costs a human annotation, and we only get a fixed budget: 6 out of 12 traces. Two strategies for spending it: send the top 6 by priority ("hard-case sampling"), or grab 6 at random. Once we spend the budget the true labels are revealed, and we can finally count real failures with `n_fail`.


In [ ]:
def n_fail(traces):
    """Count the true failures (correct == 0) in a batch of annotated traces."""
    return sum(1 for t in traces if t.correct == 0)


total = len(TRACES)
true_fail = n_fail(TRACES)  # ground truth, unknown to us until we annotate
print(f"true production failures = {true_fail}/{total} = {true_fail / total:.2f}")

budget = 6
hard = ranked[:budget]
hard_fail = n_fail(hard)
print()
print(f"Hard-case batch (top {budget} by priority): {' '.join(t.id for t in hard)}")
print(f"  failures found = {hard_fail}/{budget}  (failure density = {hard_fail / budget:.2f})")
print(f"  failure coverage = {hard_fail}/{true_fail} = {hard_fail / true_fail:.2f}")
# failures found = 4/6  (failure density = 0.67)
# failure coverage = 4/4 = 1.00


## Step 3 — The random baseline

A fixed-seed random draw stands in for "just grab some logs and hope." We use `random.Random(0)` so the batch, and therefore the count, is exactly reproducible. We also compute the *expected* number of failures a random batch of this size should contain, `budget * true_fail / total`, so the observed count has something exact to compare against, not just a hand-wave.


In [ ]:
rnd = random.Random(0).sample(TRACES, budget)
rnd = sorted(rnd, key=lambda t: t.id)
rnd_fail = n_fail(rnd)
exp_fail = budget * true_fail / total

print(f"Random batch (seed 0): {' '.join(t.id for t in rnd)}")
print(f"  failures found = {rnd_fail}/{budget}  (failure density = {rnd_fail / budget:.2f})")
print(f"  expected failures = {budget}*{true_fail}/{total} = {exp_fail:.2f}")
print(f"  failure coverage = {rnd_fail}/{true_fail} = {rnd_fail / true_fail:.2f}")
# failures found = 2/6  (failure density = 0.33)
# expected failures = 6*4/12 = 2.00
# failure coverage = 2/4 = 0.50

print()
print(f"-> Same {budget} annotations: hard-case sampling caught ALL {true_fail} failures, "
      f"random caught {rnd_fail}.")
print(f"   The golden set is {hard_fail / budget:.2f} failures vs the {true_fail / total:.2f} "
      f"base rate: ~2x denser.")


## Step 4 — The flywheel: annotate in rounds

A real annotation budget rarely arrives all at once. Split the same 6 hard-case annotations into 3 rounds of 2, and watch the golden set accumulate. Each round adds the next-most-suspicious traces (already known from the priority ranking), reveals their labels, and grows `coverage` (fraction of all real failures captured so far) and `gold rate` (fraction of the golden set so far that is a failure).


In [ ]:
print("  round  new       new_fail  golden  cum_fail  coverage  gold_rate")
golden, cum = [], 0
per_round = 2
for r in range(3):
    batch = ranked[r * per_round:(r + 1) * per_round]
    nf = n_fail(batch)
    golden += batch
    cum += nf
    ids = "+".join(t.id for t in batch)
    print(f"   {r + 1}     {ids:<8}  {nf:>7}  {len(golden):>6}  {cum:>7}   "
          f"{cum / true_fail:>6.2f}    {cum / len(golden):>6.2f}")
# round 1: T12+T11  new_fail=2  golden=2  cum_fail=2  coverage=0.50  gold_rate=1.00
# round 2: T10+T09  new_fail=1  golden=4  cum_fail=3  coverage=0.75  gold_rate=0.75
# round 3: T08+T07  new_fail=1  golden=6  cum_fail=4  coverage=1.00  gold_rate=0.67


## Step 5 — Close the loop, honestly

The golden set we just built is deliberately failure-dense, that is the entire point: it concentrates human attention on the model's weak spots instead of wasting it on traces we are already confident about. But a failure-dense set is a biased sample. Its failure rate is NOT an unbiased estimate of the true production failure rate, and reporting it as one would be misleading.


In [ ]:
gold_rate = cum / len(golden)
prod_rate = true_fail / total
print(f"golden-set failure rate = {cum}/{len(golden)} = {gold_rate:.2f}  (deliberately dense)")
print(f"true production failure rate = {true_fail}/{total} = {prod_rate:.2f}")
print()
print("The mined golden set guards the model's weak spots, but its rate is not")
print("production's rate. To estimate THAT unbiased, keep a small random holdout")
print("alongside the mined set.")
# golden-set failure rate = 4/6 = 0.67
# true production failure rate = 4/12 = 0.33


## Recap

- Production logs carry CHEAP signals (confidence, guardrail flags) even before any human annotates them. A `priority` score built from those signals is a free bet about where the failures are.
- Spending a fixed annotation budget on the highest-priority traces ("hard-case sampling") concentrates failures into the golden set far more efficiently than spending it at random: here, 4/4 real failures found in 6 annotations vs 2/4 for a random batch of the same size.
- The flywheel spends a budget in rounds: annotate the next-most-suspicious traces, grow the golden set, repeat. `coverage` (fraction of real failures captured) and `gold rate` (failure density of the golden set) both move as it grows.
- A mined golden set is deliberately failure-dense by construction, its rate is NOT the production failure rate. Keep a small random holdout alongside it to estimate the unbiased rate.

Next: taking eval scores off the offline bench and watching them run against live, unlabeled production traffic.
